# 09 — Regime-Aware Online Hybrid Ensemble

## Objective

This notebook is the final aggressive development-stage attempt to improve benchmark-relative forecasting performance before the model-development design is frozen.

The goal is **not** to force a result. The goal is to exploit genuine, time-varying predictive value while retaining Random Walk whenever more complex forecasts are unreliable.

## Expert forecasts

The online framework can use:

1. Random Walk
2. Statistical forecast from Script 05
3. Machine-Learning forecast from Script 06
4. Selective Hybrid forecast from Script 07, if available
5. Conservative RW–Statistical blends
6. Conservative RW–ML blends
7. Statistical–ML blend

This is a **hybrid model-of-models**.

## Key improvement over Script 08

Weights are updated continuously from **resolved historical errors**.

At forecast origin \(t\), only earlier forecasts whose target dates have already occurred are allowed to affect the current forecast.

### Exponentially weighted expert loss

\[
L_{m,t}
=
\sum_i \rho^{age_i}
(y_i-\hat y_{m,i})^2
\]

and expert weights are based on:

\[
w_{m,t}
\propto
\exp(-\eta \cdot RelativeLoss_{m,t})
\]

The final hybrid forecast is shrunk toward Random Walk:

\[
\hat P_{t+h}^{final}
=
P_t+
\lambda
\left(
\hat P_{t+h}^{ensemble}-P_t
\right)
\]

## Additional protections

- rolling bias correction;
- volatility/regime-sensitive trust;
- conservative fallback to RW;
- only resolved prior errors used;
- meta-configuration for each outer year selected from **earlier years only**;
- final 2024–2025 holdout is never loaded.

## Evaluation

- RMSE
- MAE
- Forecast Skill vs RW
- Theil's U
- DM p-value

In [ ]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path
import warnings
import time
import math

import numpy as np
import pandas as pd

from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 230)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
YEARS = [2019, 2020, 2021, 2022, 2023]

MIN_HISTORY = 20

# Candidate online-hybrid profiles.
# These are intentionally compact to keep runtime reasonable.
PROFILE_GRID = []

for lookback in [40, 80, 160]:
    for eta in [0.25, 0.75, 1.50]:
        for shrink in [0.25, 0.50, 0.75, 1.00]:
            PROFILE_GRID.append({
                "lookback": lookback,
                "eta": eta,
                "shrink": shrink,
                "bias_window": min(40, lookback),
                "rw_guard": True,
            })

print("Candidate online profiles:", len(PROFILE_GRID))

In [ ]:
# ============================================================
# 2. PROJECT PATHS
# ============================================================

def find_root():
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        rw = root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        stat = root/"outputs"/"predictions"/"05_statistical_predictions_2019_2023.csv"

        if rw.exists() and stat.exists():
            return root.resolve()

    raise FileNotFoundError(
        "Could not locate Script 04 and Script 05 prediction outputs."
    )

ROOT = find_root()

PRED_DIR = ROOT/"outputs"/"predictions"
TABLE_DIR = ROOT/"outputs"/"tables"

TABLE_DIR.mkdir(parents=True, exist_ok=True)

RW_PATH = PRED_DIR/"04_random_walk_predictions_2019_2023.csv"
STAT_PATH = PRED_DIR/"05_statistical_predictions_2019_2023.csv"

ML_PATHS = [
    PRED_DIR/"06_FAST_ml_predictions_2019_2023.csv",
    PRED_DIR/"06_ml_predictions_2019_2023.csv",
]

ML_PATH = next((p for p in ML_PATHS if p.exists()), None)

if ML_PATH is None:
    raise FileNotFoundError("No Script 06 ML prediction file found.")

HYBRID07_PATH = PRED_DIR/"07_hybrid_predictions_2019_2023.csv"

print("RW:", RW_PATH.name)
print("STAT:", STAT_PATH.name)
print("ML:", ML_PATH.name)
print("Hybrid07 available:", HYBRID07_PATH.exists())

In [ ]:
# ============================================================
# 3. ROBUST COLUMN DETECTION
# ============================================================

def pick_column(frame, candidates, label):
    for c in candidates:
        if c in frame.columns:
            return c

    raise ValueError(
        f"Could not find {label}. Available columns: {list(frame.columns)}"
    )


rw = pd.read_csv(
    RW_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

stat = pd.read_csv(
    STAT_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

ml = pd.read_csv(
    ML_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

stat_pred_col = pick_column(
    stat,
    [
        "Stat_Prediction",
        "Statistical_Prediction",
        "Model_Prediction",
        "Prediction",
    ],
    "statistical prediction column"
)

ml_pred_col = pick_column(
    ml,
    [
        "ML_Prediction",
        "Model_Prediction",
        "Prediction",
    ],
    "ML prediction column"
)

print("Stat column:", stat_pred_col)
print("ML column:", ml_pred_col)

In [ ]:
# ============================================================
# 4. BUILD MATCHED FORECAST PANEL
# ============================================================

KEYS = [
    "Currency",
    "Horizon",
    "Forecast_Origin",
]

base = rw[
    KEYS
    + [
        "Target_Date",
        "Actual",
        "RW_Prediction",
    ]
].copy()

base = base.merge(
    stat[
        KEYS
        + [stat_pred_col]
    ].rename(
        columns={
            stat_pred_col: "STAT"
        }
    ),
    on=KEYS,
    how="inner"
)

base = base.merge(
    ml[
        KEYS
        + [ml_pred_col]
    ].rename(
        columns={
            ml_pred_col: "ML"
        }
    ),
    on=KEYS,
    how="inner"
)

base = base.rename(
    columns={
        "RW_Prediction": "RW"
    }
)

# Optional Script 07 hybrid expert.
if HYBRID07_PATH.exists():

    hy = pd.read_csv(
        HYBRID07_PATH,
        parse_dates=["Forecast_Origin", "Target_Date"]
    )

    hy_col = pick_column(
        hy,
        [
            "Hybrid_Prediction",
            "Ensemble_Prediction",
            "Prediction",
        ],
        "Script 07 hybrid prediction column"
    )

    base = base.merge(
        hy[
            KEYS
            + [hy_col]
        ].rename(
            columns={
                hy_col: "HYBRID07"
            }
        ),
        on=KEYS,
        how="left"
    )

base["Validation_Year"] = (
    base["Forecast_Origin"].dt.year
)

base = (
    base
    .sort_values(
        [
            "Currency",
            "Horizon",
            "Forecast_Origin",
        ]
    )
    .reset_index(drop=True)
)

assert base["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")

print("Matched rows:", len(base))
display(base.head())

# 5. Create hybrid expert forecasts

We add conservative combinations as additional experts.

These are **not fitted on the evaluation outcomes**. They are predefined forecast combinations:

- `RW_STAT_25`: 75% RW + 25% Statistical
- `RW_STAT_50`: 50% RW + 50% Statistical
- `RW_ML_25`: 75% RW + 25% ML
- `RW_ML_50`: 50% RW + 50% ML
- `STAT_ML_50`: 50% Statistical + 50% ML

The online learner decides whether any of them deserve weight.

In [ ]:
# ============================================================
# 5. EXPERT LIBRARY
# ============================================================

base["RW_STAT_25"] = (
    0.75 * base["RW"]
    + 0.25 * base["STAT"]
)

base["RW_STAT_50"] = (
    0.50 * base["RW"]
    + 0.50 * base["STAT"]
)

base["RW_ML_25"] = (
    0.75 * base["RW"]
    + 0.25 * base["ML"]
)

base["RW_ML_50"] = (
    0.50 * base["RW"]
    + 0.50 * base["ML"]
)

base["STAT_ML_50"] = (
    0.50 * base["STAT"]
    + 0.50 * base["ML"]
)

EXPERTS = [
    "RW",
    "STAT",
    "ML",
    "RW_STAT_25",
    "RW_STAT_50",
    "RW_ML_25",
    "RW_ML_50",
    "STAT_ML_50",
]

if "HYBRID07" in base.columns:
    EXPERTS.append("HYBRID07")

print("Expert pool:")
print(EXPERTS)

In [ ]:
# ============================================================
# 6. METRICS
# ============================================================

def rmse(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)

    m = (
        np.isfinite(y)
        & np.isfinite(p)
    )

    if m.sum() == 0:
        return np.nan

    return float(
        np.sqrt(
            np.mean(
                (y[m]-p[m])**2
            )
        )
    )


def mae(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)

    m = (
        np.isfinite(y)
        & np.isfinite(p)
    )

    if m.sum() == 0:
        return np.nan

    return float(
        np.mean(
            np.abs(
                y[m]-p[m]
            )
        )
    )


def skill(m, r):
    if (
        not np.isfinite(m)
        or not np.isfinite(r)
        or r == 0
    ):
        return np.nan

    return float(
        1 - m/r
    )


def theils_u(m, r):
    if (
        not np.isfinite(m)
        or not np.isfinite(r)
        or r == 0
    ):
        return np.nan

    return float(
        m/r
    )


def dm_test(y, model, rw_pred, horizon=1):
    y = np.asarray(y,float)
    model = np.asarray(model,float)
    rw_pred = np.asarray(rw_pred,float)

    mask = (
        np.isfinite(y)
        & np.isfinite(model)
        & np.isfinite(rw_pred)
    )

    y = y[mask]
    model = model[mask]
    rw_pred = rw_pred[mask]

    n = len(y)

    if n < 20:
        return np.nan, np.nan

    d = (
        (y-rw_pred)**2
        - (y-model)**2
    )

    lag = max(
        0,
        int(horizon)-1
    )

    lrv = np.var(
        d,
        ddof=1
    )

    for k in range(1,lag+1):

        if len(d) <= k:
            break

        cov = np.cov(
            d[k:],
            d[:-k],
            ddof=1
        )[0,1]

        lrv += (
            2
            * (1-k/(lag+1))
            * cov
        )

    if (
        not np.isfinite(lrv)
        or lrv <= 0
    ):
        return np.nan, np.nan

    stat = (
        np.mean(d)
        / np.sqrt(lrv/n)
    )

    p = (
        2
        * student_t.sf(
            abs(stat),
            df=n-1
        )
    )

    return float(stat), float(p)

# 7. Online bias correction

For each non-RW expert, the notebook estimates recent forecast bias using **resolved past observations only**.

\[
Bias_m = mean(y-\hat y_m)
\]

and:

\[
\hat y_m^{BC} = \hat y_m + Bias_m
\]

This can remove persistent under/over-prediction without looking forward.

In [ ]:
# ============================================================
# 7. ONLINE HELPERS
# ============================================================

def resolved_history(task, origin):
    return task[
        (task["Target_Date"] <= origin)
        & (task["Forecast_Origin"] < origin)
    ].copy()


def bias_corrected_current(
    current_row,
    history,
    expert,
    bias_window
):
    pred = float(
        current_row[expert]
    )

    if expert == "RW":
        return pred

    h = history.tail(
        bias_window
    )

    if len(h) < MIN_HISTORY:
        return pred

    errors = (
        h["Actual"]
        - h[expert]
    )

    errors = errors[
        np.isfinite(errors)
    ]

    if len(errors) < MIN_HISTORY:
        return pred

    bias = float(
        errors.mean()
    )

    return pred + bias

# 8. Exponentially weighted expert losses

Recent errors receive more importance.

We normalize each expert's loss by RW loss, so the online learner focuses on **performance relative to Random Walk**.

In [ ]:
# ============================================================
# 8. EXPERT WEIGHTS
# ============================================================

def online_weights(
    history,
    profile
):
    if len(history) < MIN_HISTORY:
        return {
            e: (
                1.0
                if e == "RW"
                else 0.0
            )
            for e in EXPERTS
        }

    h = history.tail(
        int(profile["lookback"])
    ).copy()

    n = len(h)

    # More weight for recent rows.
    ages = np.arange(
        n-1,
        -1,
        -1
    )

    decay = np.exp(
        -ages
        / max(
            10.0,
            profile["lookback"]/3
        )
    )

    losses = {}

    for e in EXPERTS:

        err2 = (
            h["Actual"].to_numpy(float)
            - h[e].to_numpy(float)
        ) ** 2

        valid = np.isfinite(err2)

        if valid.sum() < MIN_HISTORY:
            losses[e] = np.inf
            continue

        w = decay[valid]

        losses[e] = float(
            np.sum(
                w * err2[valid]
            )
            / np.sum(w)
        )

    rw_loss = losses.get(
        "RW",
        np.inf
    )

    if (
        not np.isfinite(rw_loss)
        or rw_loss <= 0
    ):
        return {
            e: (
                1.0
                if e == "RW"
                else 0.0
            )
            for e in EXPERTS
        }

    raw = {}

    for e in EXPERTS:

        loss_e = losses[e]

        if not np.isfinite(loss_e):
            raw[e] = 0.0
            continue

        relative = (
            loss_e / rw_loss
        )

        # Penalize experts worse than RW exponentially.
        raw[e] = math.exp(
            -profile["eta"]
            * relative
        )

        # Optional guard:
        # strongly downweight experts clearly worse than RW.
        if (
            profile["rw_guard"]
            and relative > 1.05
        ):
            raw[e] *= 0.10

    # Give RW a stable floor.
    raw["RW"] = max(
        raw.get("RW",0.0),
        1.0
    )

    total = sum(
        raw.values()
    )

    if total <= 0:
        return {
            e: (
                1.0
                if e == "RW"
                else 0.0
            )
            for e in EXPERTS
        }

    return {
        e: raw[e]/total
        for e in EXPERTS
    }

# 9. Online hybrid simulation

The ensemble is generated sequentially.

No forecast can use an unresolved future outcome.

In [ ]:
# ============================================================
# 9. SIMULATE ONE ONLINE PROFILE
# ============================================================

def simulate_profile(
    task,
    profile,
    profile_id
):
    task = (
        task
        .sort_values(
            "Forecast_Origin"
        )
        .reset_index(drop=True)
    )

    rows = []

    for _, row in task.iterrows():

        origin = row[
            "Forecast_Origin"
        ]

        history = resolved_history(
            task,
            origin
        )

        weights = online_weights(
            history,
            profile
        )

        expert_preds = {}

        for e in EXPERTS:

            expert_preds[e] = (
                bias_corrected_current(
                    row,
                    history,
                    e,
                    int(
                        profile[
                            "bias_window"
                        ]
                    )
                )
            )

        ensemble = sum(
            weights[e]
            * expert_preds[e]
            for e in EXPERTS
        )

        # Final conservative shrinkage toward RW.
        rw_pred = float(
            row["RW"]
        )

        final = (
            rw_pred
            + profile["shrink"]
            * (
                ensemble
                - rw_pred
            )
        )

        out = {
            "Currency": row["Currency"],
            "Horizon": row["Horizon"],
            "Forecast_Origin": origin,
            "Target_Date": row["Target_Date"],
            "Validation_Year": row["Validation_Year"],
            "Actual": row["Actual"],
            "RW_Prediction": rw_pred,
            "Online_Hybrid_Prediction": final,
            "Profile_ID": profile_id,
            "Lookback": profile["lookback"],
            "Eta": profile["eta"],
            "Shrink": profile["shrink"],
            "Bias_Window": profile["bias_window"],
            "Resolved_History": len(history),
        }

        for e in EXPERTS:
            out[f"Weight_{e}"] = weights[e]

        rows.append(
            out
        )

    return pd.DataFrame(
        rows
    )

# 10. Simulate all candidate profiles

This is still much faster than refitting ML models because all underlying model forecasts already exist.

In [ ]:
# ============================================================
# 10. SIMULATE CANDIDATE PROFILES
# ============================================================

all_profiles = []

start_all = time.time()

for cur in CURRENCIES:

    for h in HORIZONS:

        task = base[
            (base["Currency"] == cur)
            & (base["Horizon"] == h)
        ].copy()

        print(
            f"Simulating {cur} H{h}...",
            flush=True
        )

        for pid, profile in enumerate(
            PROFILE_GRID
        ):

            sim = simulate_profile(
                task,
                profile,
                pid
            )

            all_profiles.append(
                sim
            )

profile_predictions = pd.concat(
    all_profiles,
    ignore_index=True
)

print(
    f"Simulation finished in "
    f"{(time.time()-start_all)/60:.2f} minutes."
)

print(
    "Rows:",
    len(profile_predictions)
)

# 11. Choose profile for each outer year using earlier years only

This is the key anti-overfitting step.

- 2019 uses a predefined conservative profile;
- 2020 profile is chosen from 2019 only;
- 2021 from 2019–2020;
- 2022 from 2019–2021;
- 2023 from 2019–2022.

The evaluated year never chooses its own profile.

In [ ]:
# ============================================================
# 11. HISTORICAL PROFILE SELECTION
# ============================================================

# Conservative default profile:
DEFAULT_PROFILE_ID = 0

selection_rows = []
final_frames = []

for cur in CURRENCIES:

    for h in HORIZONS:

        task_profiles = profile_predictions[
            (profile_predictions["Currency"] == cur)
            & (profile_predictions["Horizon"] == h)
        ].copy()

        for year in YEARS:

            if year == YEARS[0]:

                chosen_id = DEFAULT_PROFILE_ID
                historical_rmse = np.nan

            else:

                prior = task_profiles[
                    task_profiles[
                        "Validation_Year"
                    ] < year
                ].copy()

                score_rows = []

                for pid, g in prior.groupby(
                    "Profile_ID"
                ):

                    score_rows.append({
                        "Profile_ID": pid,
                        "Historical_RMSE": rmse(
                            g["Actual"],
                            g["Online_Hybrid_Prediction"]
                        )
                    })

                score_df = (
                    pd.DataFrame(
                        score_rows
                    )
                    .sort_values(
                        [
                            "Historical_RMSE",
                            "Profile_ID",
                        ]
                    )
                )

                best = score_df.iloc[0]

                chosen_id = int(
                    best["Profile_ID"]
                )

                historical_rmse = float(
                    best[
                        "Historical_RMSE"
                    ]
                )

            chosen = task_profiles[
                (task_profiles["Validation_Year"] == year)
                & (task_profiles["Profile_ID"] == chosen_id)
            ].copy()

            chosen[
                "Selected_Profile_ID"
            ] = chosen_id

            final_frames.append(
                chosen
            )

            p = PROFILE_GRID[
                chosen_id
            ]

            selection_rows.append({
                "Currency": cur,
                "Horizon": h,
                "Outer_Year": year,
                "Selected_Profile_ID": chosen_id,
                "Lookback": p["lookback"],
                "Eta": p["eta"],
                "Shrink": p["shrink"],
                "Bias_Window": p["bias_window"],
                "Historical_Selection_RMSE": historical_rmse,
            })

final_predictions = pd.concat(
    final_frames,
    ignore_index=True
)

profile_selections = pd.DataFrame(
    selection_rows
)

display(
    profile_selections
)

# 12. Yearly results

In [ ]:
# ============================================================
# 12. YEARLY RESULTS
# ============================================================

yearly = []

for (
    cur,
    h,
    year
), g in final_predictions.groupby(
    [
        "Currency",
        "Horizon",
        "Validation_Year",
    ]
):

    mr = rmse(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    ma = mae(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    rr = rmse(
        g["Actual"],
        g["RW_Prediction"]
    )

    _, p = dm_test(
        g["Actual"],
        g["Online_Hybrid_Prediction"],
        g["RW_Prediction"],
        h
    )

    yearly.append({
        "Currency": cur,
        "Horizon": int(h),
        "Validation_Year": int(year),
        "Observations": len(g),
        "RMSE": mr,
        "MAE": ma,
        "Forecast_Skill_vs_RW": skill(
            mr,
            rr
        ),
        "Theils_U": theils_u(
            mr,
            rr
        ),
        "DM_pvalue_vs_RW": p,
    })

yearly_results = pd.DataFrame(
    yearly
).sort_values(
    [
        "Currency",
        "Horizon",
        "Validation_Year",
    ]
)

display(
    yearly_results
)

yearly_results.to_csv(
    TABLE_DIR/"09_online_hybrid_yearly_results.csv",
    index=False
)

# 13. Pooled 2019–2023 results

In [ ]:
# ============================================================
# 13. POOLED RESULTS
# ============================================================

pooled = []

for (
    cur,
    h
), g in final_predictions.groupby(
    [
        "Currency",
        "Horizon",
    ]
):

    mr = rmse(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    ma = mae(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    rr = rmse(
        g["Actual"],
        g["RW_Prediction"]
    )

    _, p = dm_test(
        g["Actual"],
        g["Online_Hybrid_Prediction"],
        g["RW_Prediction"],
        h
    )

    pooled.append({
        "Currency": cur,
        "Horizon": int(h),
        "Observations": len(g),
        "RMSE": mr,
        "MAE": ma,
        "Forecast_Skill_vs_RW": skill(
            mr,
            rr
        ),
        "Theils_U": theils_u(
            mr,
            rr
        ),
        "DM_pvalue_vs_RW": p,
    })

pooled_results = pd.DataFrame(
    pooled
).sort_values(
    [
        "Currency",
        "Horizon",
    ]
)

display(
    pooled_results
)

pooled_results.to_csv(
    TABLE_DIR/"09_online_hybrid_pooled_results.csv",
    index=False
)

In [ ]:
# ============================================================
# 14. WIN SUMMARY
# ============================================================

wins = pooled_results.copy()

wins["Beats_RW"] = (
    wins["Forecast_Skill_vs_RW"] > 0
)

wins["DM_Significant_5pct"] = (
    wins["DM_pvalue_vs_RW"] < 0.05
)

display(
    wins
)

print(
    "Tasks beating RW:",
    int(
        wins[
            "Beats_RW"
        ].sum()
    ),
    "/",
    len(wins)
)

print(
    "Positive skill + DM p<0.05:",
    int(
        (
            wins["Beats_RW"]
            & wins["DM_Significant_5pct"]
        ).sum()
    ),
    "/",
    len(wins)
)

# 15. Expert-weight summary

This tells us which models the online hybrid actually trusted.

In [ ]:
# ============================================================
# 15. EXPERT WEIGHT SUMMARY
# ============================================================

weight_cols = [
    f"Weight_{e}"
    for e in EXPERTS
]

agg_dict = {
    c: "mean"
    for c in weight_cols
}

weight_summary = (
    final_predictions
    .groupby(
        [
            "Currency",
            "Horizon",
        ]
    )
    .agg(
        agg_dict
    )
    .reset_index()
)

display(
    weight_summary
)

weight_summary.to_csv(
    TABLE_DIR/"09_expert_weight_summary.csv",
    index=False
)

# 16. Save prediction-level evidence

In [ ]:
# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

final_predictions.to_csv(
    PRED_DIR/"09_online_hybrid_predictions_2019_2023.csv",
    index=False
)

profile_selections.to_csv(
    TABLE_DIR/"09_online_hybrid_profile_selections.csv",
    index=False
)

print("Saved Script 09 outputs.")

# 17. Quality gate

In [ ]:
# ============================================================
# 17. QUALITY GATE
# ============================================================

base_keys = set(
    zip(
        base["Currency"],
        base["Horizon"],
        base["Forecast_Origin"],
    )
)

final_keys = set(
    zip(
        final_predictions["Currency"],
        final_predictions["Horizon"],
        final_predictions["Forecast_Origin"],
    )
)

quality_gate = pd.DataFrame([
    {
        "Area": "Holdout isolation",
        "Status": "PASS",
        "Evidence": "Only 2019–2023 development prediction files loaded"
    },
    {
        "Area": "Hybrid model pool",
        "Status": "PASS",
        "Evidence": "RW + Statistical + ML + predefined blends + optional Script07"
    },
    {
        "Area": "Resolved-error rule",
        "Status": "PASS",
        "Evidence": "Online weights use only outcomes already known at forecast origin"
    },
    {
        "Area": "Bias correction",
        "Status": "PASS",
        "Evidence": "Rolling bias uses resolved past errors only"
    },
    {
        "Area": "Outer-year profile selection",
        "Status": "PASS",
        "Evidence": "Profile selected using earlier years only"
    },
    {
        "Area": "Matched origins",
        "Status": (
            "PASS"
            if base_keys == final_keys
            else "REVIEW"
        ),
        "Evidence": (
            f"Base={len(base_keys)}, Final={len(final_keys)}"
        )
    },
])

display(
    quality_gate
)

quality_gate.to_csv(
    TABLE_DIR/"09_quality_gate.csv",
    index=False
)

# 18. Final interpretation

A task is a numerical benchmark win when:

\[
ForecastSkill>0
\]

and:

\[
Theil's\ U<1
\]

A DM p-value below 0.05 strengthens the evidence only when the skill is positive.

After Script 09, the development design should be frozen rather than repeatedly changed to chase the pooled score.

In [ ]:
print("="*84)
print("SCRIPT 09 — REGIME-AWARE ONLINE HYBRID ENSEMBLE COMPLETE")
print("="*84)

print("\nPOOLED RESULTS:")
display(pooled_results)

print("\nPROFILE SELECTIONS:")
display(profile_selections)

print("\nAVERAGE EXPERT WEIGHTS:")
display(weight_summary)

print(
    "\nTasks beating RW:",
    int(
        (
            pooled_results[
                "Forecast_Skill_vs_RW"
            ] > 0
        ).sum()
    ),
    "/",
    len(pooled_results)
)

print("\n2024–2025 remains untouched.")